In [0]:
%python
# adjust the path to where your files landed (shown after upload)
silver = spark.read.csv("/Volumes/finance_data_platform/landing/data_ingestion/silver_financials.csv", header=True, inferSchema=True)
silver.write.mode("overwrite").saveAsTable("silver_financials")



In [0]:
%python
gold = spark.read.csv("/Volumes/finance_data_platform/landing/data_ingestion/gold_kpi_semantic.csv", header=True, inferSchema=True)
gold.write.mode("overwrite").saveAsTable("gold_kpi")

In [0]:
%sql
CREATE OR REPLACE VIEW finance_metrics
WITH METRICS
LANGUAGE YAML
AS $$
version: 1.1
comment: "Governed financial KPIs - the semantic layer"
source: gold_kpi
dimensions:
  - name: business_unit
    expr: business_unit_name
  - name: month
    expr: month
  - name: fiscal_year
    expr: fiscal_year
  - name: scenario
    expr: scenario
  - name: kpi
    expr: kpi_name
measures:
  - name: total_value
    expr: SUM(value_aud)
  - name: total_variance
    expr: SUM(variance_vs_budget)
$$;

In [0]:
%sql
SELECT business_unit, scenario, MEASURE(`total_value`) AS `total_value`, MEASURE(`total_variance`) AS `total_variance`
FROM finance_metrics
WHERE kpi = 'Total Revenue' AND month = '2026-05-01'
GROUP BY ALL
ORDER BY business_unit;

business_unit,scenario,total_value,total_variance
Eastern Region,Actual,749159,14159
Eastern Region,Budget,735000,null
Eastern Region,Forecast,722535,null
Northern Region,Actual,404530,-210470
Northern Region,Forecast,606680,null
Northern Region,Budget,615000,null
Southern Region,Actual,783481,10981
Southern Region,Budget,772500,null
Southern Region,Forecast,759170,null
Western Region,Actual,796867,9367


In [0]:
%sql
SHOW TABLES

database,tableName,isTemporary
default,finance_metrics,false
default,gold_kpi,false
default,silver_financials,false


In [0]:
display(spark.sql("SELECT DISTINCT business_unit, business_unit_name FROM silver_financials"))

business_unit,business_unit_name
BU-N,Northern Region
BU-S,Southern Region
BU-E,Eastern Region
BU-W,Western Region


In [0]:
%sql
select * from silver_financials

business_unit,cost_centre,cost_centre_name,month,fiscal_year,fiscal_month,gl_account,account_type,actual_amount_aud,transaction_count
Northern Region,CC-1010,Sales - North,2026-01-01,FY2026,7,4000,revenue,208017.57,8
Northern Region,CC-1010,Sales - North,2026-01-01,FY2026,7,5000,cost,72748.55,9
Northern Region,CC-1010,Sales - North,2026-01-01,FY2026,7,6000,cost,102878.08,8
Northern Region,CC-1010,Sales - North,2026-01-01,FY2026,7,6100,cost,34041.4,6
Northern Region,CC-1010,Sales - North,2026-01-01,FY2026,7,6200,cost,14571.99,8
Northern Region,CC-1010,Sales - North,2026-01-01,FY2026,7,6300,cost,19849.88,9
Northern Region,CC-1020,Operations - North,2026-01-01,FY2026,7,5000,cost,73063.85,9
Northern Region,CC-1020,Operations - North,2026-01-01,FY2026,7,6000,cost,96154.84,9
Northern Region,CC-1020,Operations - North,2026-01-01,FY2026,7,6100,cost,33422.88,9
Northern Region,CC-1020,Operations - North,2026-01-01,FY2026,7,6200,cost,15202.65,11


In [0]:
display(spark.sql("SHOW TABLES"))

database,tableName,isTemporary
default,finance_documents,false
default,finance_metrics,false
default,gold_kpi,false
default,silver_financials,false
